# Day 67 — CI/CD with GitHub Actions
### Automated testing (pytest), linting (Ruff), type checking (mypy), build pipelines

## 1. Setup

In [1]:
import sys
!{sys.executable} -m pip install pytest ruff mypy --quiet
print("Setup complete")

Setup complete



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. What is CI/CD?

In [2]:
cicd_concepts = """
CI/CD = Continuous Integration / Continuous Delivery

WITHOUT CI/CD:
  Developer pushes code -> hopes tests pass -> deploys manually
  -> discovers bug in production -> panic

WITH CI/CD:
  Developer pushes code -> GitHub Actions automatically:
    1. Installs dependencies
    2. Runs linting (Ruff) -- catches style/syntax errors
    3. Runs type checking (mypy) -- catches type errors
    4. Runs tests (pytest) -- catches logic errors
    5. Builds Docker image -- catches packaging errors
    6. Deploys to staging/production -- if all above pass
  -> Bug caught at step 2, 3, or 4 -- never reaches production

THE VALUE:
  "If it's not tested, it's broken" -- but tests only protect you
  if they run automatically on every change. Manual testing is
  skipped under pressure. Automated testing never is.
"""
print(cicd_concepts)

import pandas as pd
pipeline_stages = pd.DataFrame([
    {"stage": "Lint (Ruff)", "catches": "Style errors, unused imports, syntax issues", "speed": "< 1 second"},
    {"stage": "Type check (mypy)", "catches": "Type mismatches, missing attributes, wrong return types", "speed": "2-10 seconds"},
    {"stage": "Unit tests (pytest)", "catches": "Logic errors, edge cases, regression bugs", "speed": "5-60 seconds"},
    {"stage": "Integration tests", "catches": "API contract violations, database errors", "speed": "30-300 seconds"},
    {"stage": "Build (Docker)", "catches": "Dependency conflicts, missing files", "speed": "1-5 minutes"},
    {"stage": "Deploy (staging)", "catches": "Environment-specific issues", "speed": "2-10 minutes"},
])
pipeline_stages


CI/CD = Continuous Integration / Continuous Delivery

WITHOUT CI/CD:
  Developer pushes code -> hopes tests pass -> deploys manually
  -> discovers bug in production -> panic

WITH CI/CD:
  Developer pushes code -> GitHub Actions automatically:
    1. Installs dependencies
    2. Runs linting (Ruff) -- catches style/syntax errors
    3. Runs type checking (mypy) -- catches type errors
    4. Runs tests (pytest) -- catches logic errors
    5. Builds Docker image -- catches packaging errors
    6. Deploys to staging/production -- if all above pass
  -> Bug caught at step 2, 3, or 4 -- never reaches production

THE VALUE:
  "If it's not tested, it's broken" -- but tests only protect you
  if they run automatically on every change. Manual testing is
  skipped under pressure. Automated testing never is.



,stage,catches,speed
0,Lint (Ruff),"Style errors, unused imports, syntax issues",< 1 second
1,Type check (mypy),"Type mismatches, missing attributes, wrong ret...",2-10 seconds
2,Unit tests (pytest),"Logic errors, edge cases, regression bugs",5-60 seconds
3,Integration tests,"API contract violations, database errors",30-300 seconds
4,Build (Docker),"Dependency conflicts, missing files",1-5 minutes
5,Deploy (staging),Environment-specific issues,2-10 minutes


## 3. pytest — Writing Tests for ML Code

In [3]:
import os
os.makedirs("tests", exist_ok=True)

# write the module to test
ml_utils_code = '''
import numpy as np
from typing import Union

def preprocess_features(features: list[float]) -> np.ndarray:
    """Validate and preprocess input features."""
    if not features:
        raise ValueError("Features list cannot be empty")
    
    arr = np.array(features, dtype=float)
    
    if not np.all(np.isfinite(arr)):
        raise ValueError("All features must be finite (no NaN or Inf)")
    
    # normalise to 0-1 range
    min_val = arr.min()
    max_val = arr.max()
    if max_val == min_val:
        return np.zeros_like(arr)
    return (arr - min_val) / (max_val - min_val)

def compute_confidence(probability: float) -> str:
    """Convert probability to confidence label."""
    if not 0 <= probability <= 1:
        raise ValueError(f"Probability must be between 0 and 1, got {probability}")
    if probability >= 0.8:
        return "high"
    elif probability >= 0.5:
        return "medium"
    else:
        return "low"

def batch_predict(model, samples: list[list[float]]) -> list[int]:
    """Run batch prediction."""
    if not samples:
        raise ValueError("Samples list cannot be empty")
    X = np.array(samples)
    return model.predict(X).tolist()
'''

# write the test file
test_code = '''
import pytest
import numpy as np
from ml_utils import preprocess_features, compute_confidence

# ===== preprocess_features tests =====

def test_preprocess_basic():
    result = preprocess_features([0.0, 5.0, 10.0])
    expected = np.array([0.0, 0.5, 1.0])
    np.testing.assert_array_almost_equal(result, expected)

def test_preprocess_empty_raises():
    with pytest.raises(ValueError, match="cannot be empty"):
        preprocess_features([])

def test_preprocess_nan_raises():
    with pytest.raises(ValueError, match="finite"):
        preprocess_features([1.0, float("nan"), 3.0])

def test_preprocess_inf_raises():
    with pytest.raises(ValueError, match="finite"):
        preprocess_features([1.0, float("inf"), 3.0])

def test_preprocess_single_value():
    # all same values -> normalise to zeros
    result = preprocess_features([5.0, 5.0, 5.0])
    np.testing.assert_array_equal(result, [0.0, 0.0, 0.0])

def test_preprocess_negative_values():
    result = preprocess_features([-2.0, 0.0, 2.0])
    expected = np.array([0.0, 0.5, 1.0])
    np.testing.assert_array_almost_equal(result, expected)

# ===== compute_confidence tests =====

def test_confidence_high():
    assert compute_confidence(0.9) == "high"
    assert compute_confidence(0.8) == "high"
    assert compute_confidence(1.0) == "high"

def test_confidence_medium():
    assert compute_confidence(0.7) == "medium"
    assert compute_confidence(0.5) == "medium"

def test_confidence_low():
    assert compute_confidence(0.3) == "low"
    assert compute_confidence(0.0) == "low"

def test_confidence_invalid_raises():
    with pytest.raises(ValueError, match="between 0 and 1"):
        compute_confidence(1.5)
    with pytest.raises(ValueError, match="between 0 and 1"):
        compute_confidence(-0.1)

# ===== boundary tests =====

def test_confidence_boundary_08():
    assert compute_confidence(0.8) == "high"

def test_confidence_boundary_05():
    assert compute_confidence(0.5) == "medium"

def test_preprocess_two_values():
    result = preprocess_features([3.0, 7.0])
    np.testing.assert_array_almost_equal(result, [0.0, 1.0])
'''

with open("ml_utils.py", "w", encoding="utf-8") as f:
    f.write(ml_utils_code)

with open("tests/test_ml_utils.py", "w", encoding="utf-8") as f:
    f.write(test_code)

print("Files written:")
print("  ml_utils.py -- module with 3 functions")
print("  tests/test_ml_utils.py -- 13 test cases")

Files written:
  ml_utils.py -- module with 3 functions
  tests/test_ml_utils.py -- 13 test cases


## 4. Run pytest

In [4]:
import subprocess
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/test_ml_utils.py", "-v"],
    capture_output=True, text=True, cwd=os.getcwd()
)
print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr[:500])

============================= test session starts =============================
platform win32 -- Python 3.14.3, pytest-9.1.1, pluggy-1.6.0 -- c:\DS-AI-75d\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\DS-AI-75d\.vscode\week10
plugins: anyio-4.12.1, langsmith-0.9.3
collecting ... collected 13 items

tests/test_ml_utils.py::test_preprocess_basic PASSED                     [  7%]
tests/test_ml_utils.py::test_preprocess_empty_raises PASSED              [ 15%]
tests/test_ml_utils.py::test_preprocess_nan_raises PASSED                [ 23%]
tests/test_ml_utils.py::test_preprocess_inf_raises PASSED                [ 30%]
tests/test_ml_utils.py::test_preprocess_single_value PASSED              [ 38%]
tests/test_ml_utils.py::test_preprocess_negative_values PASSED           [ 46%]
tests/test_ml_utils.py::test_confidence_high PASSED                      [ 53%]
tests/test_ml_utils.py::test_confidence_medium PASSED                    [ 61%]
tests/test_ml_utils.py::test_confidence_low 

## 5. Ruff — Fast Python Linter

In [5]:
# write a file with intentional linting issues
bad_code = '''
import os
import sys
import json  # unused import

def calculate_score(x,y,z):  # missing spaces after commas
    result=x+y+z  # missing spaces around operator
    if result == True:  # should use 'if result:'
        return result
    else:
        return 0

class mlModel:  # should be MLModel (CamelCase)
    def __init__(self,name):
        self.name=name
        self.data = []

    def predict(self,X):
        return X
'''

with open("bad_code.py", "w", encoding="utf-8") as f:
    f.write(bad_code)

print("=== Running Ruff on bad_code.py ===")
result = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "bad_code.py"],
    capture_output=True, text=True
)
print(result.stdout if result.stdout else "No issues found")
if result.returncode != 0:
    print(f"Exit code: {result.returncode} (non-zero = issues found)")

print("\n=== Running Ruff on ml_utils.py (clean code) ===")
result2 = subprocess.run(
    [sys.executable, "-m", "ruff", "check", "ml_utils.py"],
    capture_output=True, text=True
)
print(result2.stdout if result2.stdout else "All checks passed -- no issues!")

=== Running Ruff on bad_code.py ===
F401 [*] `os` imported but unused
 --> bad_code.py:2:8
  |
2 | import os
  |        ^^
3 | import sys
4 | import json  # unused import
  |
help: Remove unused import: `os`

F401 [*] `sys` imported but unused
 --> bad_code.py:3:8
  |
2 | import os
3 | import sys
  |        ^^^
4 | import json  # unused import
  |
help: Remove unused import: `sys`

F401 [*] `json` imported but unused
 --> bad_code.py:4:8
  |
2 | import os
3 | import sys
4 | import json  # unused import
  |        ^^^^
5 |
6 | def calculate_score(x,y,z):  # missing spaces after commas
  |
help: Remove unused import: `json`

E712 Avoid equality comparisons to `True`; use `result:` for truth checks
  --> bad_code.py:8:8
   |
 6 | def calculate_score(x,y,z):  # missing spaces after commas
 7 |     result=x+y+z  # missing spaces around operator
 8 |     if result == True:  # should use 'if result:'
   |        ^^^^^^^^^^^^^^
 9 |         return result
10 |     else:
   |
help: Replace with 

## 6. mypy — Type Checking

In [6]:
# write a file with type errors
typed_code = '''
def add_scores(a: float, b: float) -> float:
    return a + b

def get_model_name(model_id: int) -> str:
    return model_id  # type error: returning int instead of str

def process_batch(items: list[int]) -> list[str]:
    return [str(x) for x in items]  # correct

result = add_scores("hello", 5)  # type error: str instead of float
names = process_batch([1, 2, 3])
'''

with open("typed_example.py", "w", encoding="utf-8") as f:
    f.write(typed_code)

print("=== Running mypy on typed_example.py ===")
result = subprocess.run(
    [sys.executable, "-m", "mypy", "typed_example.py", "--ignore-missing-imports"],
    capture_output=True, text=True
)
print(result.stdout)

print("=== Running mypy on ml_utils.py ===")
result2 = subprocess.run(
    [sys.executable, "-m", "mypy", "ml_utils.py", "--ignore-missing-imports"],
    capture_output=True, text=True
)
print(result2.stdout)

=== Running mypy on typed_example.py ===
typed_example.py:6: error: Incompatible return value type (got "int", expected "str")  [return-value]
typed_example.py:11: error: Argument 1 to "add_scores" has incompatible type "str"; expected "float"  [arg-type]
Found 2 errors in 1 file (checked 1 source file)

=== Running mypy on ml_utils.py ===
Success: no issues found in 1 source file



## 7. GitHub Actions Workflow — Putting It All Together

In [7]:
# write the GitHub Actions workflow
import os
os.makedirs(".github/workflows", exist_ok=True)

workflow = """name: ML Pipeline CI

on:
  push:
    branches: [ main, develop ]
  pull_request:
    branches: [ main ]

jobs:
  test:
    runs-on: ubuntu-latest
    strategy:
      matrix:
        python-version: ["3.11", "3.12"]

    steps:
    - name: Checkout code
      uses: actions/checkout@v4

    - name: Set up Python ${{ matrix.python-version }}
      uses: actions/setup-python@v5
      with:
        python-version: ${{ matrix.python-version }}

    - name: Cache pip dependencies
      uses: actions/cache@v4
      with:
        path: ~/.cache/pip
        key: ${{ runner.os }}-pip-${{ hashFiles('**/requirements.txt') }}
        restore-keys: |
          ${{ runner.os }}-pip-

    - name: Install dependencies
      run: |
        python -m pip install --upgrade pip
        pip install ruff mypy pytest
        if [ -f requirements.txt ]; then pip install -r requirements.txt; fi

    - name: Lint with Ruff
      run: ruff check . --select=E,F,W --ignore=E501

    - name: Type check with mypy
      run: mypy . --ignore-missing-imports --exclude='^(tests|setup)\\.py$'

    - name: Run tests with pytest
      run: pytest tests/ -v --tb=short

    - name: Upload test results
      uses: actions/upload-artifact@v4
      if: always()
      with:
        name: test-results-${{ matrix.python-version }}
        path: .pytest_cache/
"""

with open(".github/workflows/ml_pipeline_ci.yml", "w", encoding="utf-8") as f:
    f.write(workflow)

print("GitHub Actions workflow written to:")
print("  .github/workflows/ml_pipeline_ci.yml")
print("\nWorkflow breakdown:")
steps = [
    ("Trigger", "Runs on push to main/develop and on pull requests to main"),
    ("Matrix", "Tests on Python 3.11 AND 3.12 simultaneously"),
    ("Checkout", "actions/checkout@v4 -- clones the repo into the runner"),
    ("Setup Python", "actions/setup-python@v5 -- installs the specified Python version"),
    ("Cache pip", "Caches pip packages -- avoids reinstalling on every run"),
    ("Install deps", "pip install ruff mypy pytest + requirements.txt"),
    ("Ruff", "Linting -- fails the workflow if any issues found"),
    ("mypy", "Type checking -- fails the workflow if type errors found"),
    ("pytest", "Runs all tests -- fails if any test fails"),
    ("Upload", "Saves test results as downloadable artifact even if tests fail"),
]
for name, desc in steps:
    print(f"  {name:<15} {desc}")

GitHub Actions workflow written to:
  .github/workflows/ml_pipeline_ci.yml

Workflow breakdown:
  Trigger         Runs on push to main/develop and on pull requests to main
  Matrix          Tests on Python 3.11 AND 3.12 simultaneously
  Checkout        actions/checkout@v4 -- clones the repo into the runner
  Setup Python    actions/setup-python@v5 -- installs the specified Python version
  Cache pip       Caches pip packages -- avoids reinstalling on every run
  Install deps    pip install ruff mypy pytest + requirements.txt
  Ruff            Linting -- fails the workflow if any issues found
  mypy            Type checking -- fails the workflow if type errors found
  pytest          Runs all tests -- fails if any test fails
  Upload          Saves test results as downloadable artifact even if tests fail
